In [1]:
############################################################
#read in setup -- kind of stupid system actually
############################################################

import pandas as pd
from datetime import datetime, timedelta




with open('setup.py') as f:
    code = f.read()
exec(code)









run_sql("use hoodalgo_db")






# HARD RESET DRIVER
########################################################################################################

try:
    driver.close()
except:
    pass

try:
    driver.quit()
except:
    pass

import time, os

time.sleep(.05)

# SAFE CLEAN (won’t kill your real Chrome)
os.system("pkill -f chromedriver")
os.system("pkill -f 'chrome.*--remote-debugging-port'")
os.system("pkill -f 'chrome.*--user-data-dir=/Users/deanemarks/selenium_chrome_profile'")

time.sleep(.05)

# Recreate driverD
driver = create_driver_profile_1()

time.sleep(.05)

# Load page
file_path = "file://" + base_dir + "templates/loading_page.html"
driver.get(file_path)

driver.execute_script("document.body.style.zoom='100%'")
time.sleep(2)
########################################################################################################










############################################################
#EXECUTE in MODUELS/ FUNCTIONS -  MAKES SHARED NAMESPACE
############################################################

with open('functions.py') as f:
    code = f.read()
exec(code)






#Update Robinhood Univers Every Week
####################################################
file_path = 'robinhood_universe.csv'
created_ts = os.path.getctime(file_path)
universe_created_dt = datetime.fromtimestamp(created_ts)
one_week_ago = datetime.now() - timedelta(days=7)


if universe_created_dt > one_week_ago:
    print("Skipped Master Robinhood Universe")

if universe_created_dt < one_week_ago:
    update_loading_page(driver, "Scraping Full Robinhood Equities Universe")
    scrape_robinhood_universe(num_workers = 20)
    update_loading_page(driver, "Robinhood Equities Universe successfully Scrapped")
    time.sleep(1.5)
####################################################






✅ Deane’s MySQL Connector V39 — has query_value function 
Imported Selenium engine -- v14
📦 Switched default DB to: new_algo_db
📦 Switched default DB to: hoodalgo_db


/Users/deanemarks/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


functions read in 
Skipped Master Robinhood Universe


In [2]:
df_crypto, crypto_list = scrape_robinhood_crypto()


In [4]:
df = df_crypto


# =========================
# CLEAN + FILTER CRYPTO
# =========================

import pandas as pd

# --- STEP 1: flatten nested fields safely ---
def extract_field(obj, key):
    if isinstance(obj, dict):
        return obj.get(key)
    return None

df["asset_display_only"] = df["asset_currency"].apply(lambda x: extract_field(x, "display_only"))
df["asset_code"] = df["asset_currency"].apply(lambda x: extract_field(x, "code"))

df["quote_code"] = df["quote_currency"].apply(lambda x: extract_field(x, "code"))

# --- STEP 2: core tradable filter ---
df_tradable_crypto = df[
    (df["type"] == "cryptocurrency") &
    (df["tradability"] == "tradable") &
    (df["display_only"] == False) &
    (df["asset_display_only"] == False) &
    (df["quote_code"] == "USD")
].copy()

# --- STEP 3: remove edge case junk (important for execution) ---
df_tradable_crypto = df_tradable_crypto[
    df_tradable_crypto["max_order_size"].notna()
]

# --- STEP 4: clean output fields ---
df_tradable_crypto = df_tradable_crypto[[
    "symbol",
    "asset_code",
    "base_code",
    "base_name",
    "tradability",
    "max_order_size"
]].reset_index(drop=True)

# --- STEP 5: final list for trading ---
tradable_symbols = df_tradable_crypto["symbol"].tolist()

# =========================
# OUTPUT
# =========================

print("🔥 Tradable Crypto Count:", len(tradable_symbols))
print(tradable_symbols[:20])  # preview first 20

df_tradable_crypto

🔥 Tradable Crypto Count: 71
['CHIP-USD', 'CC-USD', 'USDG-USD', 'SKR-USD', 'LIT-USD', 'SYRUP-USD', 'AVNT-USD', 'HYPE-USD', 'XPL-USD', 'ASTER-USD', 'SEI-USD', 'ZORA-USD', 'WLFI-USD', 'XCN-USD', 'SKY-USD', 'SUI-USD', 'MOODENG-USD', 'ORCA-USD', 'MNT-USD', 'PNUT-USD']


,symbol,asset_code,base_code,base_name,tradability,max_order_size
0,CHIP-USD,CHIP,CHIP,USD.AI,tradable,400000.0000000000000000
1,CC-USD,CC,CC,Canton,tradable,1500000.0000000000000000
2,USDG-USD,USDG,USDG,Global Dollar,tradable,250000.0000000000000000
3,SKR-USD,SKR,SKR,Seeker,tradable,8000000.0000000000000000
4,LIT-USD,LIT,LIT,Lighter,tradable,100000.0000000000000000
...,...,...,...,...,...,...
66,ETC-USD,ETC,ETC,Ethereum Classic,tradable,10000.0000000000000000
67,DOGE-USD,DOGE,DOGE,Dogecoin,tradable,6500000.0000000000000000
68,BCH-USD,BCH,BCH,Bitcoin Cash,tradable,750.0000000000000000
69,ETH-USD,ETH,ETH,Ethereum,tradable,280.0000000000000000
